# MTUNet — Match Them Up: Visually Explainable Few-Shot Image Classification

This notebook runs the full MTUNet pipeline on Google Colab:
1. Environment setup & GPU check
2. Clone repo & install dependencies
3. Download & prepare dataset (CIFAR-FS)
4. Stage 1 — Train backbone (ResNet-18)
5. Stage 2 — Train Scouter (slot attention patterns)
6. Stage 3 — Train FSL model (few-shot learning)
7. Evaluate on test set
8. Visualize attention maps

> **Paper**: *Match Them Up: Visually Explainable Few-shot Image Classification* (CVPR 2021 Workshop + Applied Intelligence 2022)  
> **GitHub**: https://github.com/wbw520/MTUNet

## 1. Check GPU & Runtime

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be extremely slow on CPU.")
    print("Go to Runtime → Change runtime type → GPU to enable GPU acceleration.")

In [ ]:
# Show disk space available
!df -h /content

## 2. Clone Repository & Install Dependencies

In [ ]:
import os

REPO_DIR = "/content/MTUNet"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/wbw520/MTUNet.git {REPO_DIR}
else:
    print("Repo already cloned, pulling latest changes...")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

In [ ]:
# ── Fix transform_func.py for modern torchvision ──────────────────────────
# Older code uses Image.BILINEAR (deprecated PIL constant); newer torchvision
# requires InterpolationMode enum. Patch the file in-place after cloning.
import pathlib, re

tf_path = pathlib.Path("/content/MTUNet/loaders/datasets/transform_func.py")
src = tf_path.read_text()

if "InterpolationMode" not in src:
    src = src.replace(
        "import torchvision.transforms.functional as F",
        "import torchvision.transforms.functional as F\nfrom torchvision.transforms import InterpolationMode",
    )
    # Replace old PIL constant defaults with InterpolationMode
    src = src.replace("interpolation=Image.BILINEAR", "interpolation=InterpolationMode.BILINEAR")
    # Remove the _pil_interpolation_to_str dict (references removed Image.* constants)
    src = re.sub(
        r"\n_pil_interpolation_to_str = \{.*?\}\n\n",
        "\n\n",
        src,
        flags=re.DOTALL,
    )
    # Fix __repr__ that used the removed dict
    src = src.replace(
        "        interpolate_str = _pil_interpolation_to_str[self.interpolation]\n"
        "        return self.__class__.__name__ + '(size={0}, interpolation={1})'.format(self.size, interpolate_str)",
        "        return self.__class__.__name__ + '(size={0}, interpolation={1})'.format(self.size, self.interpolation)",
    )
    tf_path.write_text(src)
    print("transform_func.py patched for modern torchvision.")
else:
    print("transform_func.py already patched.")

In [ ]:
# Install dependencies
!pip install -q \
    imgaug==0.4.0 \
    prefetch-generator==1.0.1 \
    thop \
    tensorly \
    scikit-image \
    tqdm \
    opencv-python

# ── NumPy 2.0 compatibility patch for imgaug ──────────────────────────────
# imgaug 0.4.0 hard-codes np.sctypes which was removed in NumPy 2.0.
# We patch the installed source file directly so every subprocess
# (train_base.py, train_scouter.py, etc.) also benefits.
import pathlib, re

imgaug_py = pathlib.Path("/usr/local/lib/python3.12/dist-packages/imgaug/imgaug.py")
src = imgaug_py.read_text()

PATCH = """\
# -- numpy 2.0 compat patch (auto-applied) --
import numpy as _np
if not hasattr(_np, 'sctypes'):
    _np.sctypes = {
        'int':     [_np.int8, _np.int16, _np.int32, _np.int64],
        'uint':    [_np.uint8, _np.uint16, _np.uint32, _np.uint64],
        'float':   [_np.float16, _np.float32, _np.float64],
        'complex': [_np.complex64, _np.complex128],
        'others':  [bool, object, bytes, str],
    }
# -- end patch --
"""

if "numpy 2.0 compat patch" not in src:
    # Insert patch right after the first 'import numpy' line
    src = re.sub(r'(import numpy as np\n)', r'\1' + PATCH, src, count=1)
    imgaug_py.write_text(src)
    print("imgaug.py patched for NumPy 2.0 compatibility.")
else:
    print("Patch already applied.")

print("All dependencies ready.")

In [ ]:
# Verify key imports
# np.sctypes patch must already be applied by the cell above before importing imgaug
import numpy as np
import cv2
import matplotlib
import sklearn
import tqdm
import imgaug
import PIL

print(f"numpy       {np.__version__}")
print(f"opencv      {cv2.__version__}")
print(f"matplotlib  {matplotlib.__version__}")
print(f"sklearn     {sklearn.__version__}")
print(f"tqdm        {tqdm.__version__}")
print(f"imgaug      {imgaug.__version__}")
print(f"Pillow      {PIL.__version__}")
print("\nAll imports OK!")

## 3. Dataset — Download & Prepare CIFAR-FS

CIFAR-FS is derived from CIFAR-100 and split into 64 train / 16 val / 20 test classes.

We download CIFAR-100 via `torchvision`, extract it, and re-organise into the directory structure expected by MTUNet.

In [ ]:
import torchvision
import pickle
import shutil
from pathlib import Path

DATA_ROOT = Path("/content/FSL_data")
CIFAR100_RAW = DATA_ROOT / "cifar100_raw"
CIFARFS_ROOT = DATA_ROOT / "cifarfs"

DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Download CIFAR-100 via torchvision (downloads to CIFAR100_RAW)
print("Downloading CIFAR-100...")
ds = torchvision.datasets.CIFAR100(root=str(CIFAR100_RAW), download=True)
print("Download complete.")
!ls {CIFAR100_RAW}

In [ ]:
# CIFAR-FS official splits (Bertinetto et al.)
# 64 train / 16 val / 20 test fine-grained classes of CIFAR-100

CIFARFS_SPLITS = {
    "train": [
        "apple", "aquarium_fish", "baby", "bear", "beaver", "bed", "bee", "beetle",
        "bicycle", "bottle", "bowl", "boy", "bridge", "bus", "butterfly", "camel",
        "can", "castle", "caterpillar", "cattle", "chair", "chimpanzee", "clock",
        "cloud", "cockroach", "couch", "crab", "crocodile", "cup", "dinosaur",
        "dolphin", "elephant", "flatfish", "forest", "fox", "girl", "hamster",
        "house", "kangaroo", "keyboard", "lamp", "lawn_mower", "leopard", "lion",
        "lizard", "lobster", "man", "maple_tree", "motorcycle", "mountain",
        "mouse", "mushroom", "oak_tree", "orange", "orchid", "otter", "palm_tree",
        "pear", "pickup_truck", "pine_tree", "plain", "plate", "poppy", "porcupine"
    ],
    "val": [
        "possum", "rabbit", "raccoon", "ray", "road", "rocket", "rose",
        "sea", "seal", "shark", "shrew", "skunk", "skyscraper", "snail", "snake", "spider"
    ],
    "test": [
        "squirrel", "streetcar", "sunflower", "sweet_pepper", "table", "tank",
        "telephone", "television", "tiger", "tractor", "train", "trout",
        "tulip", "turtle", "wardrobe", "whale", "willow_tree", "wolf", "woman", "worm"
    ]
}

print("Train classes:", len(CIFARFS_SPLITS['train']))
print("Val   classes:", len(CIFARFS_SPLITS['val']))
print("Test  classes:", len(CIFARFS_SPLITS['test']))

In [ ]:
import numpy as np
from PIL import Image
import csv

def save_cifar100_as_images(raw_root, out_root):
    """Convert CIFAR-100 binary files → class-organized PNG folders."""
    raw_root = Path(raw_root) / "cifar-100-python"
    out_root = Path(out_root)

    # Load meta to get class names
    with open(raw_root / "meta", "rb") as f:
        meta = pickle.load(f, encoding="bytes")
    fine_label_names = [n.decode() for n in meta[b"fine_label_names"]]

    for split_file, split_name in [("train", "train"), ("test", "test")]:
        with open(raw_root / split_file, "rb") as f:
            data = pickle.load(f, encoding="bytes")

        images = data[b"data"]          # (N, 3072)
        fine_labels = data[b"fine_labels"]
        filenames = [fn.decode() for fn in data[b"filenames"]]

        for img_flat, label_idx, fname in zip(images, fine_labels, filenames):
            class_name = fine_label_names[label_idx]
            class_dir = out_root / class_name
            class_dir.mkdir(parents=True, exist_ok=True)
            img = img_flat.reshape(3, 32, 32).transpose(1, 2, 0)  # HWC
            Image.fromarray(img).save(class_dir / fname)

    print(f"Images written to {out_root}")
    return fine_label_names


IMAGES_DIR = DATA_ROOT / "cifar100_images"

if not IMAGES_DIR.exists():
    print("Extracting CIFAR-100 images (this takes ~2-3 min)...")
    fine_names = save_cifar100_as_images(CIFAR100_RAW, IMAGES_DIR)
else:
    print("Images already extracted, skipping.")
    with open(CIFAR100_RAW / "cifar-100-python" / "meta", "rb") as f:
        meta = pickle.load(f, encoding="bytes")
    fine_names = [n.decode() for n in meta[b"fine_label_names"]]

print(f"Total classes in image dir: {len(list(IMAGES_DIR.iterdir()))}")

In [ ]:
def build_cifarfs_split(images_dir, cifarfs_root, split_name, class_list):
    """Copy class folders into cifarfs/{split}/ and write the CSV."""
    split_dir = cifarfs_root / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    csv_dir = Path("/content/MTUNet/data/data_split/cifarfs")
    csv_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for cls in class_list:
        src = images_dir / cls
        dst = split_dir / cls
        if not dst.exists():
            shutil.copytree(str(src), str(dst))
        for img_file in dst.iterdir():
            rows.append([f"{cls}/{img_file.name}", cls])

    csv_path = csv_dir / f"{split_name}.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filename", "label"])
        writer.writerows(rows)

    print(f"{split_name}: {len(class_list)} classes, {len(rows)} images → {csv_path}")


if not CIFARFS_ROOT.exists():
    print("Building CIFAR-FS splits...")
    for split, classes in CIFARFS_SPLITS.items():
        build_cifarfs_split(IMAGES_DIR, CIFARFS_ROOT, split, classes)
    print("CIFAR-FS dataset ready.")
else:
    print("CIFAR-FS already prepared, skipping.")

!ls {DATA_ROOT}/cifarfs

In [ ]:
# Verify CSV files exist
!ls /content/MTUNet/data/data_split/cifarfs/
!head -5 /content/MTUNet/data/data_split/cifarfs/train.csv

In [ ]:
# Quick visual check — show a few sample images from different classes
import matplotlib.pyplot as plt
import random
from PIL import Image

fig, axes = plt.subplots(3, 8, figsize=(16, 6))
fig.suptitle("Sample images from CIFAR-FS (train split)", fontsize=13)

sample_classes = random.sample(CIFARFS_SPLITS["train"], 8)

for col, cls in enumerate(sample_classes):
    cls_dir = CIFARFS_ROOT / "train" / cls
    imgs = list(cls_dir.iterdir())[:3]
    for row, img_path in enumerate(imgs):
        axes[row][col].imshow(Image.open(img_path))
        axes[row][col].axis("off")
        if row == 0:
            axes[row][col].set_title(cls, fontsize=8)

plt.tight_layout()
plt.show()

## 4. Configuration

All hyperparameters are set here. Change them before running training cells.

In [ ]:
import sys
sys.path.insert(0, "/content/MTUNet")
os.chdir("/content/MTUNet")

# ─── Dataset ──────────────────────────────────────────────────────────────────
DATASET      = "cifarfs"       # miniImageNet | tiered-ImageNet | cifarfs | CUB200
DATA_ROOT_S  = "/content/FSL_data"
NUM_CLASSES  = 64              # 64 for cifarfs/miniImageNet, 351 for tiered-ImageNet

# ─── Backbone ─────────────────────────────────────────────────────────────────
BASE_MODEL   = "resnet18"      # resnet18 | wideres | conv4 | mobilenet | densenet
CHANNEL      = 512             # 512 for resnet18, 640 for wideres
IMG_SIZE     = 80

# ─── FSL episode setup ────────────────────────────────────────────────────────
N_WAY        = 5
N_SHOT       = 1               # 1-shot or 5-shot
QUERY        = 15

# ─── Slot / Scouter ───────────────────────────────────────────────────────────
INTERVAL     = 10              # sample every N classes for patterns
NUM_SLOT     = NUM_CLASSES // INTERVAL   # = 6 for cifarfs with interval=10

# ─── Training ─────────────────────────────────────────────────────────────────
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE   = 128
LR           = 0.001
EPOCHS_BASE  = 20              # backbone training epochs
EPOCHS_SLOT  = 30              # scouter training epochs
EPOCHS_FSL   = 20              # FSL training epochs

TRAIN_EPS    = 200             # episodes per training epoch
VAL_EPS      = 500             # episodes for validation
TEST_EPS     = 1000            # episodes for final test

print("Configuration:")
print(f"  Dataset     : {DATASET}")
print(f"  Backbone    : {BASE_MODEL}")
print(f"  Device      : {DEVICE}")
print(f"  N-way/K-shot: {N_WAY}-way {N_SHOT}-shot")
print(f"  Num slots   : {NUM_SLOT}")

## 5. Stage 1 — Train Backbone

Trains a ResNet-18 backbone on all 64 base classes using standard cross-entropy.
The trained backbone weights are saved and reused in Stage 2 & 3.

In [ ]:
!python train_base.py \
    --dataset {DATASET} \
    --data_root {DATA_ROOT_S} \
    --base_model {BASE_MODEL} \
    --channel {CHANNEL} \
    --num_classes {NUM_CLASSES} \
    --img_size {IMG_SIZE} \
    --epochs {EPOCHS_BASE} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --n_way {N_WAY} \
    --n_shot {N_SHOT} \
    --query {QUERY} \
    --val_episodes {VAL_EPS} \
    --device {DEVICE} \
    --num_workers 2 \
    --aug True \
    --use_slot False

In [ ]:
# Confirm checkpoint saved
!ls -lh saved_model/

## 6. Stage 2 — Train Scouter (Slot Attention Patterns)

Trains the slot-based attention module on top of the frozen backbone.
The scouter learns to assign visual features to discriminative pattern slots.

In [ ]:
!python train_scouter.py \
    --dataset {DATASET} \
    --data_root {DATA_ROOT_S} \
    --base_model {BASE_MODEL} \
    --channel {CHANNEL} \
    --num_classes {NUM_CLASSES} \
    --img_size {IMG_SIZE} \
    --epochs {EPOCHS_SLOT} \
    --batch_size {BATCH_SIZE} \
    --lr 0.0001 \
    --lr_drop 20 \
    --interval {INTERVAL} \
    --num_slot {NUM_SLOT} \
    --random False \
    --use_slot True \
    --loss_status 1 \
    --hidden_dim 64 \
    --slots_per_class 1 \
    --power 2 \
    --to_k_layer 3 \
    --lambda_value "1." \
    --device {DEVICE} \
    --num_workers 2

In [ ]:
!ls -lh saved_model/

## 7. Stage 3 — Train FSL Model (Few-Shot Learning)

Fine-tunes the full MTUNet model (backbone + scouter) using episodic few-shot training.
Each episode is a 5-way 1-shot task drawn from the training classes.

In [ ]:
!python train_fsl.py \
    --dataset {DATASET} \
    --data_root {DATA_ROOT_S} \
    --base_model {BASE_MODEL} \
    --channel {CHANNEL} \
    --num_classes {NUM_CLASSES} \
    --img_size {IMG_SIZE} \
    --epochs {EPOCHS_FSL} \
    --lr 0.0001 \
    --lr_drop 10 \
    --n_way {N_WAY} \
    --n_shot {N_SHOT} \
    --query {QUERY} \
    --train_episodes {TRAIN_EPS} \
    --val_episodes {VAL_EPS} \
    --interval {INTERVAL} \
    --num_slot {NUM_SLOT} \
    --random False \
    --use_slot True \
    --loss_status 1 \
    --hidden_dim 64 \
    --slots_per_class 1 \
    --power 2 \
    --to_k_layer 3 \
    --lambda_value "1." \
    --device {DEVICE} \
    --num_workers 2

In [ ]:
!ls -lh saved_model/

## 8. Evaluate on Test Set

Runs 1000 random 5-way 1-shot episodes on the held-out test classes and reports mean accuracy ± 95% confidence interval.

In [ ]:
!python test_fsl.py \
    --dataset {DATASET} \
    --data_root {DATA_ROOT_S} \
    --base_model {BASE_MODEL} \
    --channel {CHANNEL} \
    --num_classes {NUM_CLASSES} \
    --img_size {IMG_SIZE} \
    --n_way {N_WAY} \
    --n_shot {N_SHOT} \
    --query {QUERY} \
    --test_episodes {TEST_EPS} \
    --interval {INTERVAL} \
    --num_slot {NUM_SLOT} \
    --random False \
    --use_slot True \
    --hidden_dim 64 \
    --slots_per_class 1 \
    --power 2 \
    --to_k_layer 3 \
    --lambda_value "1." \
    --device {DEVICE} \
    --num_workers 2

## 9. Visualize Attention Maps

Generates slot-attention heatmaps overlaid on query images.
These show *which image regions* the model uses for its classification decision.

In [ ]:
!python vis_fsl.py \
    --dataset {DATASET} \
    --data_root {DATA_ROOT_S} \
    --base_model {BASE_MODEL} \
    --channel {CHANNEL} \
    --num_classes {NUM_CLASSES} \
    --img_size {IMG_SIZE} \
    --n_way {N_WAY} \
    --n_shot {N_SHOT} \
    --query 1 \
    --interval {INTERVAL} \
    --num_slot {NUM_SLOT} \
    --random False \
    --use_slot True \
    --vis True \
    --vis_id 0 \
    --hidden_dim 64 \
    --slots_per_class 1 \
    --power 2 \
    --to_k_layer 3 \
    --lambda_value "1." \
    --device {DEVICE} \
    --num_workers 0

In [ ]:
# Show the generated visualizations
import glob
import matplotlib.pyplot as plt
from PIL import Image

vis_paths = sorted(glob.glob("vis/**/*.png", recursive=True))[:16]

if not vis_paths:
    print("No visualization images found. Check that vis_fsl.py ran successfully.")
else:
    cols = 4
    rows = (len(vis_paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = axes.flatten()

    for ax, path in zip(axes, vis_paths):
        ax.imshow(Image.open(path))
        ax.set_title(Path(path).name, fontsize=7)
        ax.axis("off")

    for ax in axes[len(vis_paths):]:
        ax.axis("off")

    fig.suptitle("MTUNet Slot-Attention Visualizations", fontsize=14)
    plt.tight_layout()
    plt.show()
    print(f"Displayed {len(vis_paths)} visualization(s).")

## 10. (Optional) Run 5-shot Evaluation

Re-run Stages 3 and 8 with `N_SHOT=5` for the 5-shot accuracy comparison.

In [ ]:
N_SHOT_5 = 5

print("=== Training FSL model for 5-shot ===")
!python train_fsl.py \
    --dataset {DATASET} \
    --data_root {DATA_ROOT_S} \
    --base_model {BASE_MODEL} \
    --channel {CHANNEL} \
    --num_classes {NUM_CLASSES} \
    --img_size {IMG_SIZE} \
    --epochs {EPOCHS_FSL} \
    --lr 0.0001 \
    --lr_drop 10 \
    --n_way {N_WAY} \
    --n_shot {N_SHOT_5} \
    --query {QUERY} \
    --train_episodes {TRAIN_EPS} \
    --val_episodes {VAL_EPS} \
    --interval {INTERVAL} \
    --num_slot {NUM_SLOT} \
    --random False \
    --use_slot True \
    --loss_status 1 \
    --hidden_dim 64 \
    --slots_per_class 1 \
    --power 2 \
    --to_k_layer 3 \
    --lambda_value "1." \
    --device {DEVICE} \
    --num_workers 2

In [ ]:
print("=== Testing 5-shot ===")
!python test_fsl.py \
    --dataset {DATASET} \
    --data_root {DATA_ROOT_S} \
    --base_model {BASE_MODEL} \
    --channel {CHANNEL} \
    --num_classes {NUM_CLASSES} \
    --img_size {IMG_SIZE} \
    --n_way {N_WAY} \
    --n_shot {N_SHOT_5} \
    --query {QUERY} \
    --test_episodes {TEST_EPS} \
    --interval {INTERVAL} \
    --num_slot {NUM_SLOT} \
    --random False \
    --use_slot True \
    --hidden_dim 64 \
    --slots_per_class 1 \
    --power 2 \
    --to_k_layer 3 \
    --lambda_value "1." \
    --device {DEVICE} \
    --num_workers 2

## 11. Save Results to Google Drive (Optional)

Mounts your Drive and copies checkpoints + visualizations there so they survive a Colab session reset.

In [ ]:
# Uncomment and run this cell to save to Drive

# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil
# SAVE_DIR = "/content/drive/MyDrive/MTUNet_results"
# shutil.copytree("/content/MTUNet/saved_model", SAVE_DIR + "/saved_model", dirs_exist_ok=True)
# shutil.copytree("/content/MTUNet/vis",         SAVE_DIR + "/vis",         dirs_exist_ok=True)
# print("Results saved to Google Drive:", SAVE_DIR)